# 🌾 EDA – Barley · Focus 10 dernières années

In [ ]:
import sys, os

# Add project root to sys.path so 'constants' is importable
PROJECT_ROOT = "/Users/gregzguegue/Desktop/DSB/24_Data_for_Strat_BCG/BCG-Data-for-Strategy"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from constants.path import SILVER_PATH

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


In [ ]:
df_all = pd.read_parquet("data/processed/silver/barley.parquet")

last_year = df_all["year"].max()
first_year_10 = last_year - 9  # 10 years inclusive
df = df_all[df_all["year"] >= first_year_10].copy()

print(f"Période retenue : {first_year_10} → {last_year}  ({df['year'].nunique()} années)")
print(f"Départements    : {df['department'].nunique()}")
print(f"Lignes          : {len(df)}")
df.head()


---
## 1 · Tendances fortes sur 10 ans
Pour chaque métrique (**yield**, **production**, **area**) : quels départements ont le plus augmenté / diminué ?

In [ ]:
def compute_growth(df, metric, first_year, last_year, n=5):
    """Return top-n growers and top-n decliners for `metric`."""
    first = (
        df[df["year"] == first_year]
        .groupby("department")[metric].mean()
        .rename("first")
    )
    last = (
        df[df["year"] == last_year]
        .groupby("department")[metric].mean()
        .rename("last")
    )
    g = pd.concat([first, last], axis=1).dropna()
    g["growth_pct"] = ((g["last"] - g["first"]) / g["first"]) * 100
    g = g.sort_values("growth_pct", ascending=False)
    return g.head(n), g.tail(n), g


metrics_labels = {
    "yield":      "Rendement (t/ha)",
    "production": "Production (t)",
    "area":       "Surface cultivée (ha)",
}

growth_data = {}
for metric, label in metrics_labels.items():
    top, bot, full = compute_growth(df, metric, first_year_10, last_year, n=5)
    growth_data[metric] = {"top": top, "bot": bot, "full": full}
    print(f"\n{'═'*60}")
    print(f"  📊  {label}")
    print(f"{'═'*60}")
    print(f"\n  🔼  Top 5 – Plus forte hausse :")
    for dep, row in top.iterrows():
        print(f"      {dep:25s}  {row['growth_pct']:+7.1f}%   "
              f"({row['first']:,.0f} → {row['last']:,.0f})")
    print(f"\n  🔽  Top 5 – Plus forte baisse :")
    for dep, row in bot.iterrows():
        print(f"      {dep:25s}  {row['growth_pct']:+7.1f}%   "
              f"({row['first']:,.0f} → {row['last']:,.0f})")


In [ ]:
# Visualisation – barres horizontales pour chaque métrique
fig, axes = plt.subplots(3, 2, figsize=(18, 14))

colors_up   = "#16A34A"
colors_down = "#DC2626"

for i, (metric, label) in enumerate(metrics_labels.items()):
    top = growth_data[metric]["top"]
    bot = growth_data[metric]["bot"]

    # Top hausse
    ax = axes[i, 0]
    ax.barh(top.index[::-1], top["growth_pct"][::-1], color=colors_up,
            edgecolor="white")
    ax.set_title(f"🔼  {label} – Plus forte hausse", fontweight="bold")
    ax.set_xlabel("Croissance (%)")
    for bar, val in zip(ax.patches, top["growth_pct"][::-1]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"+{val:.1f}%", va="center", fontsize=10, fontweight="bold")

    # Top baisse
    ax = axes[i, 1]
    ax.barh(bot.index[::-1], bot["growth_pct"][::-1], color=colors_down,
            edgecolor="white")
    ax.set_title(f"🔽  {label} – Plus forte baisse", fontweight="bold")
    ax.set_xlabel("Croissance (%)")
    for bar, val in zip(ax.patches, bot["growth_pct"][::-1]):
        ax.text(bar.get_width() - 0.5, bar.get_y() + bar.get_height()/2,
                f"{val:.1f}%", va="center", ha="right", fontsize=10,
                fontweight="bold", color="white")

fig.suptitle("Évolution sur 10 ans – Hausse vs Baisse par métrique",
             fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()


---
## 2 · Plus gros producteurs & plus fiables (10 dernières années)
- **Plus grosse production** = production moyenne la plus élevée
- **Plus fiable** = coefficient de variation (CV) du rendement le plus bas (rendement stable d'une année à l'autre)

In [ ]:
dep_stats = df.groupby("department").agg(
    avg_production=("production", "mean"),
    avg_yield=("yield", "mean"),
    std_yield=("yield", "std"),
    avg_area=("area", "mean"),
).dropna()
dep_stats["cv_yield"] = (dep_stats["std_yield"] / dep_stats["avg_yield"]) * 100

# ── Top producteurs ──
top10_prod = dep_stats.nlargest(10, "avg_production")
print("🏭  Top 10 – Production moyenne (t) sur 10 ans :")
for i, (dep, row) in enumerate(top10_prod.iterrows(), 1):
    print(f"  {i:>2}. {dep:25s}  {row['avg_production']:>12,.0f} t")

# ── Plus fiables (CV bas) ──
# On ne considère que les départements avec une production significative
# (dans le top 50 %) pour éviter les très petits producteurs biaisés
median_prod = dep_stats["avg_production"].median()
significant = dep_stats[dep_stats["avg_production"] >= median_prod]
top10_reliable = significant.nsmallest(10, "cv_yield")
print(f"\n🎯  Top 10 – Plus fiables (CV rendement le plus bas, prod ≥ médiane) :")
for i, (dep, row) in enumerate(top10_reliable.iterrows(), 1):
    print(f"  {i:>2}. {dep:25s}  CV = {row['cv_yield']:5.1f}%"
          f"   (rdt moy = {row['avg_yield']:.2f} t/ha)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Top 10 production
ax = axes[0]
ax.barh(top10_prod.index[::-1], top10_prod["avg_production"][::-1],
        color="#2563EB", edgecolor="white")
ax.set_title("Top 10 – Production moyenne (t)", fontweight="bold")
ax.set_xlabel("Production moyenne (t)")
for bar, val in zip(ax.patches, top10_prod["avg_production"][::-1]):
    ax.text(bar.get_width() * 0.98, bar.get_y() + bar.get_height()/2,
            f"{val:,.0f}", va="center", ha="right", fontsize=10,
            fontweight="bold", color="white")

# Top 10 fiabilité
ax = axes[1]
ax.barh(top10_reliable.index[::-1], top10_reliable["cv_yield"][::-1],
        color="#16A34A", edgecolor="white")
ax.set_title("Top 10 – Plus fiables (CV rendement %)", fontweight="bold")
ax.set_xlabel("Coefficient de variation du rendement (%)")
ax.invert_xaxis()
for bar, val in zip(ax.patches, top10_reliable["cv_yield"][::-1]):
    ax.text(bar.get_width() * 1.02, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", ha="left", fontsize=10,
            fontweight="bold")

fig.tight_layout()
plt.show()


---
## 3 · Focus ClientCo – 10 dernières années
Départements : **Essonne, Somme, Cher, Haute-Garonne, Isère**.

In [ ]:
CLIENTCO = ["Essonne", "Somme", "Cher"]
df_cc = df[df["department"].isin(CLIENTCO)].copy()
print(f"Lignes ClientCo : {len(df_cc)}")
df_cc.head()


### 3.1 · Tableau récapitulatif des KPI

In [ ]:
kpi = df_cc.groupby("department").agg(
    rdt_moy=("yield", "mean"),
    rdt_std=("yield", "std"),
    prod_moy=("production", "mean"),
    prod_tot=("production", "sum"),
    area_moy=("area", "mean"),
).round(2)
kpi["cv_yield_%"] = ((kpi["rdt_std"] / kpi["rdt_moy"]) * 100).round(1)

# Growth rate per metric
for metric in ["yield", "production", "area"]:
    first = df_cc[df_cc["year"] == first_year_10].set_index("department")[metric]
    last  = df_cc[df_cc["year"] == last_year].set_index("department")[metric]
    kpi[f"{metric}_growth_%"] = (((last - first) / first) * 100).round(1)

kpi = kpi.reindex(CLIENTCO)
kpi.style.format(precision=1).background_gradient(cmap="RdYlGn", axis=0)


### 3.2 · Évolution du rendement

In [ ]:
colors_cc = ["#2563EB", "#16A34A", "#F59E0B", "#DC2626", "#8B5CF6"]

fig, ax = plt.subplots(figsize=(14, 6))
for dep, color in zip(CLIENTCO, colors_cc):
    sub = df_cc[df_cc["department"] == dep].sort_values("year")
    ax.plot(sub["year"], sub["yield"], linewidth=2.2, marker="o",
            markersize=5, color=color, label=dep)

ax.set_title("Rendement annuel – Zones ClientCo (10 dernières années)",
             fontweight="bold")
ax.set_xlabel("Année")
ax.set_ylabel("Rendement (t/ha)")
ax.legend(frameon=True, fancybox=True, shadow=True)
fig.tight_layout()
plt.show()


### 3.3 · Évolution de la production

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for dep, color in zip(CLIENTCO, colors_cc):
    sub = df_cc[df_cc["department"] == dep].sort_values("year")
    ax.plot(sub["year"], sub["production"], linewidth=2.2, marker="s",
            markersize=5, color=color, label=dep)

ax.set_title("Production annuelle – Zones ClientCo (10 dernières années)",
             fontweight="bold")
ax.set_xlabel("Année")
ax.set_ylabel("Production (t)")
ax.legend(frameon=True, fancybox=True, shadow=True)
fig.tight_layout()
plt.show()


### 3.4 · Évolution de la surface cultivée

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for dep, color in zip(CLIENTCO, colors_cc):
    sub = df_cc[df_cc["department"] == dep].sort_values("year")
    ax.plot(sub["year"], sub["area"], linewidth=2.2, marker="^",
            markersize=5, color=color, label=dep)

ax.set_title("Surface cultivée – Zones ClientCo (10 dernières années)",
             fontweight="bold")
ax.set_xlabel("Année")
ax.set_ylabel("Surface (ha)")
ax.legend(frameon=True, fancybox=True, shadow=True)
fig.tight_layout()
plt.show()


### 3.5 · Heatmap – Rendement par année & département

In [ ]:
pivot_yield = df_cc.pivot_table(index="department", columns="year",
                                 values="yield", aggfunc="mean")
pivot_yield = pivot_yield.reindex(CLIENTCO)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot_yield, annot=True, fmt=".1f", cmap="RdYlGn",
            linewidths=0.5, linecolor="white", ax=ax, cbar_kws={"label": "t/ha"})
ax.set_title("Heatmap du rendement (t/ha) – Zones ClientCo",
             fontweight="bold")
ax.set_ylabel("")
fig.tight_layout()
plt.show()


### 3.6 · Positionnement – Production vs Rendement vs Fiabilité

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    kpi["rdt_moy"],
    kpi["prod_moy"],
    s=(100 - kpi["cv_yield_%"]) * 15,   # plus fiable = plus gros point
    c=colors_cc[:len(kpi)],
    alpha=0.85, edgecolors="white", linewidth=2,
)

for dep, row in kpi.iterrows():
    ax.annotate(
        f"{dep}\nCV={row['cv_yield_%']:.0f}%",
        (row["rdt_moy"], row["prod_moy"]),
        textcoords="offset points", xytext=(12, 8),
        fontsize=10, fontweight="bold",
    )

ax.set_title("Production moy. vs Rendement moy.\n"
             "(taille ∝ fiabilité, CV bas = gros point)", fontweight="bold")
ax.set_xlabel("Rendement moyen (t/ha)")
ax.set_ylabel("Production moyenne (t)")
fig.tight_layout()
plt.show()
